In [25]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import pandas as pd

In [16]:
X_train = pd.read_csv('csv/predictors_scaled_train.csv')
X_test = pd.read_csv('csv/predictors_scaled_test.csv')
y_train = pd.read_csv('csv/target_train.csv')
y_test = pd.read_csv('csv/target_test.csv')

In [17]:
regions_train = X_train[['region']]
regions_test = X_test[['region']]
X_train = X_train.drop('region', axis=1)
X_test = X_test.drop('region', axis=1)
y_train = y_train.drop('region', axis=1)
y_test = y_test.drop('region', axis=1)

In [7]:
ridge_regressor = Ridge()
lasso_regressor = Lasso()

In [5]:
grid_params = {
    'alpha': [0.01, 0.05, 0.1, 0.5, 1, 10]
}

# Ridge Regression (L2-регуляризация)

## Поиск оптимального коэффициента регуляризации по сетке с помощью кросс-валидации

In [18]:
grid_ridge = GridSearchCV(ridge_regressor, grid_params, cv=5, verbose=1, scoring='r2')
grid_ridge.fit(X_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.01, 0.05, 0.1, 0.5, 1, 10]}, scoring='r2',
             verbose=1)

In [22]:
ridge_regressor = grid_ridge.best_estimator_
ridge_params = grid_ridge.best_params_

#### Лучший подобранный коэффициент регуляризации

In [23]:
ridge_params['alpha']

10

### Предсказанные значения

In [56]:
y_pred = ridge_regressor.predict(X_test)
y_train_pred = ridge_regressor.predict(X_train)

In [39]:
response_test = pd.concat([regions_test, pd.Series(y_pred, name='ratio_underage_partic_predicted'), y_test], axis=1)
response_train = pd.concat([regions_train, pd.Series(y_train_pred, name='ratio_underage_partic_predicted'), y_train], axis=1)

In [40]:
response_test

,region,ratio_underage_partic_predicted,ratio_underage_partic
0,Приморский край,0.644171,0.703667
1,Белгородская область,0.295284,0.243133
2,Красноярский край,0.674703,0.587017
3,Вологодская область,0.594626,0.591933
4,Смоленская область,0.576519,0.596300
5,Сахалинская область,0.754003,0.826383
6,Орловская область,0.434570,0.301567
7,Республика Карелия,0.640388,0.919950
8,Ивановская область,0.485390,0.500083
9,Республика Хакасия,0.604988,0.574600


In [41]:
response_train

,region,ratio_underage_partic_predicted,ratio_underage_partic
0,Камчатский край,0.592471,0.723200
1,Чеченская Республика,0.009479,0.003583
2,Чувашская Республика - Чувашия,0.298737,0.339983
3,Рязанская область,0.276937,0.253967
4,Республика Калмыкия,0.397390,0.266633
...,...,...,...
63,Ненецкий автономный округ,0.622945,0.709033
64,Ханты-Мансийский автономный округ - Югра,0.339708,0.215500
65,Новосибирская область,0.578752,0.619767
66,Тверская область,0.590171,0.495067


### Метрики

In [57]:
ridge_metrics = pd.DataFrame({
    'MAE':[mean_absolute_error(y_pred, y_test), mean_absolute_error(y_train_pred, y_train)],
    'MAPE':[mean_absolute_percentage_error(y_pred, y_test), mean_absolute_percentage_error(y_train_pred, y_train)],
    'R2':[r2_score(y_pred, y_test), r2_score(y_train_pred, y_train)]
}, index = ['test', 'train'])

In [58]:
ridge_metrics

,MAE,MAPE,R2
test,0.080699,0.209822,0.545603
train,0.079934,0.184972,0.656715


### Коэффициенты

In [48]:
coeffs = pd.DataFrame(ridge_regressor.coef_, index=X_train.columns, columns=['coeffs'])
coeffs

,coeffs
health_ratio,-0.008994
coeff_Gini,-0.014189
not_smoking_male%,-0.062051
not_smoking_female%,-0.076342
num_of_criminals,-0.005477
num_of_alcohol_crimes,0.001238
num_of_female_crimes,-0.014703
num_of_drugs_crimes,0.006099
num_of_underaged_criminals,0.076980
life_quality,-0.080904


# Lasso Regression (L1-регуляризация)

## Поиск оптимального коэффициента регуляризации по сетке с помощью кросс-валидации

In [50]:
grid_lasso = GridSearchCV(lasso_regressor, grid_params, cv=5, scoring='r2', verbose=1)
grid_lasso.fit(X_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


GridSearchCV(cv=5, estimator=Lasso(),
             param_grid={'alpha': [0.01, 0.05, 0.1, 0.5, 1, 10]}, scoring='r2',
             verbose=1)

In [51]:
lasso_regressor = grid_lasso.best_estimator_
lasso_params = grid_lasso.best_params_

#### Оптимальный коэффициент регуляризации

In [53]:
lasso_params['alpha']

0.01

### Предсказанные значения

In [59]:
y_pred = lasso_regressor.predict(X_test)
y_train_pred = lasso_regressor.predict(X_train)

In [61]:
response_test = pd.concat([regions_test, pd.Series(y_pred, name='ratio_underage_partic_predicted'), y_test], axis=1)
response_train = pd.concat([regions_train, pd.Series(y_train_pred, name='ratio_underage_partic_predicted'), y_train], axis=1)

In [62]:
response_test

,region,ratio_underage_partic_predicted,ratio_underage_partic
0,Приморский край,0.638246,0.703667
1,Белгородская область,0.288785,0.243133
2,Красноярский край,0.699101,0.587017
3,Вологодская область,0.579057,0.591933
4,Смоленская область,0.574643,0.596300
5,Сахалинская область,0.761782,0.826383
6,Орловская область,0.426621,0.301567
7,Республика Карелия,0.631097,0.919950
8,Ивановская область,0.478875,0.500083
9,Республика Хакасия,0.581732,0.574600


In [63]:
response_train

,region,ratio_underage_partic_predicted,ratio_underage_partic
0,Камчатский край,0.590673,0.723200
1,Чеченская Республика,0.005275,0.003583
2,Чувашская Республика - Чувашия,0.289051,0.339983
3,Рязанская область,0.266490,0.253967
4,Республика Калмыкия,0.375474,0.266633
...,...,...,...
63,Ненецкий автономный округ,0.652151,0.709033
64,Ханты-Мансийский автономный округ - Югра,0.336773,0.215500
65,Новосибирская область,0.557687,0.619767
66,Тверская область,0.577046,0.495067


### Метрики

In [60]:
lasso_metrics = pd.DataFrame({
    'MAE':[mean_absolute_error(y_pred, y_test), mean_absolute_error(y_train_pred, y_train)],
    'MAPE':[mean_absolute_percentage_error(y_pred, y_test), mean_absolute_percentage_error(y_train_pred, y_train)],
    'R2':[r2_score(y_pred, y_test), r2_score(y_train_pred, y_train)]
}, index = ['test', 'train'])
lasso_metrics

,MAE,MAPE,R2
test,0.085775,0.221241,0.501010
train,0.081627,0.187228,0.649349


### Коэффициенты

In [64]:
coeffs = pd.DataFrame(lasso_regressor.coef_, index=X_train.columns, columns=['coeffs'])
coeffs

,coeffs
health_ratio,-0.000000
coeff_Gini,-0.003969
not_smoking_male%,-0.064590
not_smoking_female%,-0.076291
num_of_criminals,-0.000000
num_of_alcohol_crimes,-0.000000
num_of_female_crimes,-0.000000
num_of_drugs_crimes,-0.000000
num_of_underaged_criminals,0.068324
life_quality,-0.092600
